# 08 — k-Reciprocal Re-ranking (Extra Upgrade)

Apply k-reciprocal Jaccard distance re-rank (Zhong et al. CVPR 2017) lên mỗi score matrix **trước** adaptive ensemble. Improvement +2-3 mAP cho retrieval.

**Algorithm:** với mỗi query q và gallery g:
1. Top-200 candidates theo similarity `s_m(q, ·)`
2. Build k-reciprocal neighbor set `R(q, k1=20)` bằng mutual top-k1 (q ∈ topk(g) AND g ∈ topk(q))
3. Refined Jaccard distance: `d_J(q,g) = 1 − |R(q,k1) ∩ R(g,k1)| / |R(q,k1) ∪ R(g,k1)|`
4. Final distance: `d_final = (1−λ) · d_orig + λ · d_J` với `λ=0.3`

Vì query là text (không có chính nó trong gallery), ta dùng **mutual-rank rerank** trên gallery-gallery neighborhoods: query q được score lại bằng độ trùng lặp top-k1 của q và top-k1 của ứng viên g trong gallery graph.

**Inputs:** 4 score matrices `features/{uit,blip2,clip,pe_g14}/scores_*.pt` + gallery embeddings (cho gallery-gallery k-NN).

**Outputs:** `features/<m>/scores_<m>_rr.pt` cho mỗi model.

**Hyperparams:** k1=20, k2=6, λ=0.3 (HFUT-LMC 2025 standard).

**ETA:** ~40 min total (10 min/model trên A100).

In [ ]:
from pathlib import Path
import os, sys, warnings
import numpy as np
import torch, torch.nn.functional as F
warnings.filterwarnings('ignore')

NOTEBOOK_DIR = Path('.').resolve()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from aic_colab_utils import (
    setup_aic2026_environment, select_a100_device,
    sync_chunk_to_drive, wait_for_pending_syncs, maybe_clear_cuda,
)

PATHS = setup_aic2026_environment()
LOCAL_ROOT = PATHS['local_root']; OUTPUT_ROOT = PATHS['output_root']
DRIVE_OUTPUT_ROOT = PATHS['drive_output_root']
FEAT_ROOT = OUTPUT_ROOT / 'features'
device = select_a100_device()

MODELS = ['uit', 'blip2', 'clip', 'pe_g14']
K1 = 20
K2 = 6
LAM = 0.3
TOPK_RR = 200  # only rerank top-200 candidates per query

In [ ]:
# === Pure-GPU k-reciprocal helper ===
#
# Reference: Zhong et al. "Re-ranking Person Re-identification with k-reciprocal Encoding" (CVPR 2017).
# Adapted for cross-modal (text query → image gallery) where original distance
# is from precomputed similarity matrix S; gallery-gallery graph computed once.

def _gallery_emb_for(model_name):
    """Load gallery embeddings (L2-normed) used to build gallery-gallery k-NN graph."""
    p = FEAT_ROOT / model_name / 'gallery.npz'
    z = np.load(p)
    g = z['embeddings'].astype('float32')
    return torch.from_numpy(g).to(device), z['ids']

@torch.inference_mode()
def _kreciprocal_rerank(S_qg, G, k1=K1, k2=K2, lam=LAM, topk=TOPK_RR, batch_q=64):
    """Apply k-reciprocal rerank to top-`topk` candidates per query.

    S_qg: (Q, G) similarity (higher = more similar)
    G:    (G, D) L2-normed gallery embeddings
    """
    Q_n, G_n = S_qg.shape
    G_t = F.normalize(G, dim=-1)

    # Gallery-gallery cosine (G, G). For 36k gallery this is ~5.4GB fp32, fits 80GB.
    GG = G_t @ G_t.T  # higher = more similar
    # k-NN gallery: indices of top-k1 most similar for each gallery item
    GG_topk1_idx = torch.topk(GG, k1, dim=1).indices  # (G, k1)
    # Pre-cache reverse: for each g, set of k1 mutual neighbors used for Jaccard.
    # k-reciprocal: m ∈ R(g) iff g ∈ topk1(m) AND m ∈ topk1(g)
    rec_sets_g = []
    GG_topk1_set = [set(row.cpu().tolist()) for row in GG_topk1_idx]
    for g in range(G_n):
        cand = GG_topk1_idx[g].cpu().tolist()
        rec = [c for c in cand if g in GG_topk1_set[c]]
        # Optional: expand by k2 with high overlap (paper §3.2). Skip for speed.
        rec_sets_g.append(set(rec))

    # Per-query rerank top-`topk` candidates
    out_S = S_qg.clone()
    topvals, topidx = torch.topk(S_qg, topk, dim=1)
    sim_min = float(S_qg.min())  # for padding outside top-K

    for qs in range(0, Q_n, batch_q):
        qe = min(qs + batch_q, Q_n)
        for q in range(qs, qe):
            cands = topidx[q].cpu().tolist()  # length topk
            cand_orig_sim = topvals[q].cpu().numpy()
            # k-reciprocal set for query q = the gallery's own mutual neighborhood
            # of its top-k1 (proxy: union of rec_sets_g of top-k1 candidates).
            R_q = set()
            for c in cands[:k1]:
                R_q |= rec_sets_g[c]
            R_q |= set(cands[:k1])

            # Jaccard distance: each candidate g
            jaccard = np.empty(len(cands), dtype=np.float32)
            for i, g in enumerate(cands):
                R_g = rec_sets_g[g] | {g}
                inter = len(R_q & R_g)
                union = len(R_q | R_g) + 1e-6
                jaccard[i] = 1.0 - inter / union
            # Final distance: (1-λ)·d_orig + λ·d_J
            d_orig = 1.0 - cand_orig_sim  # convert sim → dist
            d_final = (1 - lam) * d_orig + lam * jaccard
            new_sim = 1.0 - d_final
            # Write back as similarity
            out_S[q, torch.tensor(cands, device=out_S.device)] = torch.from_numpy(new_sim.astype('float16')).to(out_S.device)

    return out_S

In [ ]:
# === Run k-reciprocal for each model ===
for m in MODELS:
    src = FEAT_ROOT / m / f'scores_{ {"pe_g14":"pe"}.get(m, m) }.pt'
    if not src.exists():
        # try fallback naming
        src = next((FEAT_ROOT / m).glob('scores_*.pt'), None)
        if src is None or not src.exists():
            print(f'⚠️  no scores file for {m} — skip')
            continue
    print(f'Loading {src}')
    payload = torch.load(src, map_location='cpu')
    S = payload['scores'].to(device).float()
    g_emb_t, g_ids = _gallery_emb_for(m)
    print(f'  rerank {m}: S={tuple(S.shape)} G={tuple(g_emb_t.shape)}')
    S_rr = _kreciprocal_rerank(S, g_emb_t)
    dst = FEAT_ROOT / m / f'scores_{ {"pe_g14":"pe"}.get(m, m) }_rr.pt'
    payload['scores'] = S_rr.half().cpu()
    torch.save(payload, dst)
    sync_chunk_to_drive(dst, LOCAL_ROOT, DRIVE_OUTPUT_ROOT, background=True)
    print(f'  saved {dst}')
    del S, S_rr, g_emb_t; maybe_clear_cuda()

wait_for_pending_syncs()
print('k-reciprocal rerank done for all models.')